# Simon's Problem

Given a function $f:\{0,1\}^n \to \{0,1\}^m$ promised to be either one-to-one, or two-to-one with a hidden string $s \neq 0^n$ such that

$$f(x) = f(y) \iff x = y \quad\text{or}\quad x = y \oplus s$$

determine $s$.

## Classical difficulty

To find $s$ classically you must locate a *collision* — two distinct inputs $x$ and $x'$ with $f(x) = f(x')$. Once found, $s = x \oplus x'$.

A natural first instinct is to test each bit position independently: hold all bits at 0 and flip bit $i$, checking whether $f(0\cdots0) = f(0\cdots1\cdots0)$. The problem is that this only detects $s = e_i$ — the single-bit string with only position $i$ set. In our $n = 3$ example with $s = 110$, flipping only bit 1 gives a different output ($f(000) = 101$ but $f(100) = 000$), and flipping only bit 2 also gives a different output ($f(010) = 000$) — yet flipping *both* bits 1 and 2 together gives the same output ($f(110) = 101 = f(000)$). No single-bit test reveals anything useful; you have to find the right *combination* of bits to flip, and there are $2^n - 1$ non-zero combinations to try.

In practice the only general strategy is to query $f$ at a sequence of inputs and watch for two that produce the same output.

**Worst case (deterministic).** Imagine an adversary who answers your queries consistently but tries to delay revealing the collision as long as possible. For each input you query, the adversary can place it in the "first of its pair" — returning a fresh output value you have never seen before — as long as fewer than half the inputs have been queried. The adversary can sustain this for your first $2^{n-1}$ queries, giving $2^{n-1}$ distinct outputs with no collision. Only on query $2^{n-1} + 1$ does the pigeonhole principle force a repeated output. In the worst case you therefore need $2^{n-1} + 1$ queries — exponential in $n$.

**Worst case (randomised).** If you query random inputs, the *birthday paradox* applies: with $k$ random draws from $2^n$ inputs, the probability of a collision grows as roughly $1 - e^{-k^2/2^{n+1}}$. To reach a constant probability of success you need $k = \Omega(2^{n/2})$ queries — still exponential, and the best any classical randomised algorithm can achieve.

Simon's algorithm solves the problem with $O(n)$ quantum queries — an exponential improvement.

## Eliminating the one-to-one case

Before running the full algorithm it is worth asking: can we cheaply rule out the one-to-one case? If $f$ is one-to-one, every input maps to a unique output and there is no collision to find — $s$ does not exist. Accidentally treating a one-to-one function as if it had a period would give nonsensical results.

The simplest check is to evaluate $f$ at a small number of randomly chosen inputs and compare outputs. If any two inputs $x$ and $x'$ produce the same output, the function is immediately confirmed two-to-one and $s = x \oplus x'$ is read off directly — no quantum computation needed at all. This "lucky collision" shortcut is always worth trying first.

If no collision appears after a handful of probes, you cannot yet conclude the function is one-to-one: you may simply have been unlucky. As the analysis above shows, even distinguishing one-to-one from two-to-one with certainty requires exponentially many classical queries. This is the gap that Simon's algorithm closes: after $O(n)$ quantum runs the Gaussian elimination either yields a unique non-zero $s$ (confirming two-to-one) or produces only the trivial solution $s = 0\cdots0$ (confirming one-to-one), with no exponential search required.

## Example: $n = 3$, $s = 110$

The following function (from the Wikipedia article on Simon's problem) has period $s = 110$. Every output value appears exactly twice, and the two inputs that share an output always differ by $s$:

<table style="border-collapse:collapse;font-family:monospace">
  <tr><th style="padding:4px 16px;border:1px solid #aaa">x</th><th style="padding:4px 16px;border:1px solid #aaa">f(x)</th></tr>
  <tr style="background:#cce5ff"><td style="padding:4px 16px;border:1px solid #aaa">000</td><td style="padding:4px 16px;border:1px solid #aaa">101</td></tr>
  <tr style="background:#d4edda"><td style="padding:4px 16px;border:1px solid #aaa">001</td><td style="padding:4px 16px;border:1px solid #aaa">010</td></tr>
  <tr style="background:#fff3cd"><td style="padding:4px 16px;border:1px solid #aaa">010</td><td style="padding:4px 16px;border:1px solid #aaa">000</td></tr>
  <tr style="background:#f8d7da"><td style="padding:4px 16px;border:1px solid #aaa">011</td><td style="padding:4px 16px;border:1px solid #aaa">110</td></tr>
  <tr style="background:#fff3cd"><td style="padding:4px 16px;border:1px solid #aaa">100</td><td style="padding:4px 16px;border:1px solid #aaa">000</td></tr>
  <tr style="background:#f8d7da"><td style="padding:4px 16px;border:1px solid #aaa">101</td><td style="padding:4px 16px;border:1px solid #aaa">110</td></tr>
  <tr style="background:#cce5ff"><td style="padding:4px 16px;border:1px solid #aaa">110</td><td style="padding:4px 16px;border:1px solid #aaa">101</td></tr>
  <tr style="background:#d4edda"><td style="padding:4px 16px;border:1px solid #aaa">111</td><td style="padding:4px 16px;border:1px solid #aaa">010</td></tr>
</table>

Rows sharing a background colour are pairs: each pair satisfies $f(x) = f(x \oplus 110)$. For instance, $f(000) = f(110) = 101$ ✓ and $f(011) = f(101) = 110$ ✓.

This is the same as saying $x \oplus 110$ is the other input that produces the same output as $x$. Because XOR is its own inverse — XORing both sides with $x$ turns $f(x) = f(x \oplus s)$ back into itself with the two inputs swapped — the relationship is symmetric: each input's partner is exactly $x \oplus s$.

The third output bit, $f_3$, is $1$ exactly when $x_1 = x_2$ and $x_3 = 0$ — an AND-type condition that cannot be built from CNOTs alone (CNOTs can only produce XOR/linear combinations of the inputs). The oracle therefore uses an ancilla qubit `t` and a Toffoli (CCNOT) gate:

1. Flip `t` to $|1\rangle$, then CNOT $a \to t$ and CNOT $b \to t$, giving $t = 1 \oplus x_1 \oplus x_2$ (the XNOR of $x_1, x_2$).
2. CNOT $t \to d$, then CNOT $c \to d$, giving $f_1 = d = 1 \oplus x_1 \oplus x_2 \oplus x_3$.
3. CNOT $c \to e$, giving $f_2 = e = x_3$.
4. CNOT $t \to f$, then Toffoli($t, c \to f$), giving $f_3 = f = t \wedge \lnot x_3 = (1 \oplus x_1 \oplus x_2) \wedge (1 \oplus x_3)$ — the Toffoli supplies the AND that a CNOT-only circuit cannot.

<img src="images/simons-oracle-circuit.svg" style="max-width: 100%; height: auto;" alt="Oracle circuit for n=3, s=110: an ancilla qubit t and a Toffoli gate implementing f(x)=(1 xor x1 xor x2 xor x3, x3, (1 xor x1 xor x2) and (1 xor x3))">

The circuit above is exactly the oracle's gate structure, run on a computational-basis input instead of a superposition. Below, the oracle is verified directly on each of the four colliding pairs the table above requires: rather than checking classical inputs one at a time, each pair is prepared as a genuine superposition $(|x\rangle + |x \oplus s\rangle)/\sqrt2$ and the circuit is confirmed to collapse it to the single shared output value.</cell id="sp01md00">


In [1]:
from qubit import Qubit
from gates_single import hadamard, qnot
from gates_multi import cnot, toffoli
from sympy.physics.paulialgebra import Pauli


def measure_z(z_observable, bits):
    """Substitute a computational-basis input into an evolved Z observable
    and read off the resulting bit. Z = +1 for |0>, Z = -1 for |1>; a qubit
    prepared in a definite basis state has X = Y = 0."""
    substitutions = {}
    for label, bit in bits.items():
        substitutions[Pauli(1, label=label)] = 0
        substitutions[Pauli(2, label=label)] = 0
        substitutions[Pauli(3, label=label)] = 1 - 2 * bit
    return 0 if z_observable.subs(substitutions) == 1 else 1


def oracle(qa, qb, qc, qd, qe, qf, qt):
    """t = 1 xor x1 xor x2, d = t xor x3, e = x3, f = t and not(x3)."""
    qt = qnot(qt)
    qa, qt = cnot(qa, qt)
    qb, qt = cnot(qb, qt)
    qt, qd = cnot(qt, qd)
    qc, qd = cnot(qc, qd)
    qc, qe = cnot(qc, qe)
    qt, qf = cnot(qt, qf)
    qt, qc, qf = toffoli(qt, qc, qf)
    return qa, qb, qc, qd, qe, qf, qt


def collision_output(b, c):
    """Prepare (|0,b,c> + |1,not b,c>)/sqrt(2) -- a pair that differs by
    s=110 -- run it through the oracle, and return its (definite) output."""
    qa, qb, qc = Qubit.qubit_time_0('a'), Qubit.qubit_time_0('b'), Qubit.qubit_time_0('c')
    qd, qe, qf = (Qubit.qubit_time_0(label) for label in 'def')
    qt = Qubit.qubit_time_0('t')

    if b:
        qb = qnot(qb)
    if c:
        qc = qnot(qc)
    qa = hadamard(qa)
    qa, qb = cnot(qa, qb)

    qa, qb, qc, qd, qe, qf, qt = oracle(qa, qb, qc, qd, qe, qf, qt)

    # The Heisenberg-evolved output observables are evaluated on the original
    # |0...0> state, which is exactly the state actually prepared above (the
    # Hadamard and Bell-pairing CNOT are already baked into qd.z, qe.z, qf.z).
    # If the oracle is correct, both branches of the superposition collide on
    # the same output, so this collapses to a single definite value.
    bits = {label: 0 for label in 'abcdeft'}
    return tuple(measure_z(q.z, bits) for q in (qd, qe, qf))


# Every collision demanded by s = 110: x differs from x XOR 110 only in
# whether (x1,x2) is 00 vs 11 or 01 vs 10, with x3 held fixed.
pairs = [
    (0, 0, "000", "110", "101"),
    (0, 1, "001", "111", "010"),
    (1, 0, "010", "100", "000"),
    (1, 1, "011", "101", "110"),
]

for b, c, x, xp, expected in pairs:
    actual = collision_output(b, c)
    output = ''.join(map(str, actual))
    assert output == expected, f"f({x})=f({xp}) should give {expected}, got {output}"
    print(f"|{x}> + |{xp}> -> |{output}>")

print("\nAll 4 collisions confirmed -- each output is a single definite value even "
      "though a and b remain in superposition, exactly as required by s = 110.")

|000> + |110> -> |101>
|001> + |111> -> |010>
|010> + |100> -> |000>
|011> + |101> -> |110>

All 4 collisions confirmed -- each output is a single definite value even though a and b remain in superposition, exactly as required by s = 110.


## The Algorithm

Each run of the circuit produces an $n$-bit measurement from the input qubits. The results are not random — they carry a hidden structure that reflects $s$. Certain measurement outcomes are completely suppressed when a bit of $s$ is 1, while others can only appear when a bit of $s$ is 0. Running the circuit a small number of times and comparing the pattern of results is enough to pin down every bit of $s$. The intuition section below explains exactly why.

### Generic algorithm structure

For an arbitrary $n$-bit function $f$, the circuit has $n$ input qubits (labeled $x_1, \ldots, x_n$) and $n$ output qubits (labeled $y_1, \ldots, y_n$). The oracle $U_f$ implements the function-specific logic.

<img src="images/simons-generic-circuit.svg" style="max-width: 100%; height: auto;" alt="Generic Simon's algorithm circuit diagram">

Steps:
1. Apply H to every input qubit.
2. Apply the oracle $U_f$, which maps $|x\rangle|y\rangle \mapsto |x\rangle|y \oplus f(x)\rangle$.
3. Apply H to every input qubit again.
4. Measure the input qubits.


## Intuition

It helps to imagine measuring the output qubits immediately after the oracle, even though the algorithm does not require it. In this example the output measures as $|110\rangle$, which — looking at the truth table — means the input must have been either $011$ or $101$.

<img src="images/simons-intuition-circuit.svg" style="max-width: 100%; height: auto;" alt="Simon's algorithm circuit showing intermediate state">

**State interpretation at each stage:**

Right after the oracle (before the final Hadamards), the output qubits have measured as:
- $d = |1\rangle$, $e = |1\rangle$, $f = |0\rangle$ (that is, $f(x) = 110$)

This constrains the input to be either $011$ or $101$. Each input qubit's state reflects the ambiguity:

- **Qubit `a`** ($s_1 = 1$): the candidates $011$ and $101$ have $a = 0$ and $a = 1$ — they differ, so `a` is in superposition $|0\rangle + |1\rangle$.
- **Qubit `b`** ($s_2 = 1$): the candidates have $b = 1$ and $b = 0$ — they differ, so `b` is in superposition $|1\rangle + |0\rangle$.
- **Qubit `c`** ($s_3 = 0$): both candidates have $c = 1$ — they agree, so `c` is in the definite state $|1\rangle$.

After the final Hadamard gates, the input qubits are measured:
- Qubits where $s_i = 1$ (those in superposition: `a`, `b`) always measure to the same value: either both 0 or both 1.
- Qubits where $s_i = 0$ (those in definite states: `c`) measure 0 or 1 with equal probability.

**General pattern:** A qubit at position $i$ where $s_i = 1$ always ends up in superposition after the oracle, because the secret string flips that bit between the two candidate inputs. A qubit where $s_i = 0$ ends up in a definite state, because the secret string does not flip that bit.

The second set of Hadamard gates converts this into a readable signal:

- $H|{+}\rangle = |0\rangle$ — a qubit that was in superposition **always measures 0**. This identifies positions where $s_i = 1$.
- $H|0\rangle = |{+}\rangle$ and $H|1\rangle = |{-}\rangle$ — a qubit that was in a definite state **measures 0 or 1 with equal probability**. This identifies positions where $s_i = 0$.

Any input qubit that ever measures **1** must therefore have $s_i = 0$. A qubit that never measures 1 across many runs very likely has $s_i = 1$. Collecting measurements across runs reveals the bit pattern of $s$.

> **Note on entanglement.** When multiple positions have $s_i = 1$ — as here, where both `a` and `b` have $s_i = 1$ — those qubits are entangled with each other after the oracle, not independently in state $|{+}\rangle$. Their individual measurement statistics are 50/50, but they are correlated: they always produce the same outcome. This means you can never see `a` measure 1 while `b` measures 0 or vice versa.

In [2]:
from qubit import Qubit
from gates_single import hadamard
from IPython.display import display

# Input qubits (x₁, x₂, x₃) — s = 110 means s₁=1, s₂=1, s₃=0
qa = Qubit.qubit_time_0('a')
qb = Qubit.qubit_time_0('b')
qc = Qubit.qubit_time_0('c')

# Output qubits — not measured, but entangled by the oracle
qd = Qubit.qubit_time_0('d')
qe = Qubit.qubit_time_0('e')
qf = Qubit.qubit_time_0('f')

# Ancilla — needed because f₃ is an AND-type (nonlinear) function of the
# inputs, which no CNOT-only circuit can produce
qt = Qubit.qubit_time_0('t')

# Step 1: Hadamard on all input qubits
qa = hadamard(qa)
qb = hadamard(qb)
qc = hadamard(qc)

# Step 2: Oracle  f(x₁,x₂,x₃) = (1⊕x₁⊕x₂⊕x₃, x₃, (1⊕x₁⊕x₂)∧(1⊕x₃))
# This function has period s=110: flipping x₁ and x₂ together leaves the output unchanged.
# (the `oracle` function is defined above, in the collision-verification cell)
qa, qb, qc, qd, qe, qf, qt = oracle(qa, qb, qc, qd, qe, qf, qt)

# Step 3: Hadamard on all input qubits
qa = hadamard(qa)
qb = hadamard(qb)
qc = hadamard(qc)

display(qa)
display(qb)
display(qc)

Qubit(a, a1, a2*t1, a3*t1)

Qubit(b, b1, b2*t1, b3*t1)

Qubit(c, c1, -(-a1*b1*c2*d1*e1*f1*t3/2 + a1*b1*c2*d1*e1*t3/2 - c2*d1*e1/2 - c2*d1*e1*f1/2), a1*b1*c3*d1*e1*f1*t3/2 - a1*b1*c3*d1*e1*t3/2 + c3*d1*e1/2 + c3*d1*e1*f1/2)

## Reading the result

The Z observable of each input qubit after the circuit determines its measurement statistics.

**Qubit `a`** has $Z = \sigma_z^{(a)}\,\sigma_x^{(t)}$. The factor $\sigma_x^{(t)}$ has expectation value $0$ in the initial state, so `a` measures 0 or 1 with equal probability — but $\sigma_x^{(t)}$ is *also* the shared factor in `b`'s Z observable.

**Qubit `b`** has $Z = \sigma_z^{(b)}\,\sigma_x^{(t)}$ — the same $\sigma_x^{(t)}$. The product $Z_a \cdot Z_b = \sigma_z^{(a)}\,\sigma_z^{(b)}\,(\sigma_x^{(t)})^2 = \sigma_z^{(a)}\,\sigma_z^{(b)}$ has expectation $+1$, which means `a` and `b` always give the **same** measurement outcome. This is the observable expression of $y_1 = y_2$, i.e. $s_1 = s_2 = 1$.

**Qubit `c`** is more intricate than `a` and `b`, because `c` feeds into the Toffoli gate that builds the nonlinear $f_3$. Its evolved observable factors as

$$Z_c = \sigma_z^{(c)}\sigma_x^{(d)}\sigma_x^{(e)}\left[\frac{1+\sigma_x^{(f)}}{2} + \sigma_x^{(a)}\sigma_x^{(b)}\sigma_z^{(t)}\cdot\frac{\sigma_x^{(f)}-1}{2}\right].$$

On the $\sigma_x^{(f)} = +1$ branch this reduces to the same clean shape as before, $\sigma_z^{(c)}\sigma_x^{(d)}\sigma_x^{(e)}$; on the $\sigma_x^{(f)} = -1$ branch it instead picks up $-\sigma_z^{(c)}\sigma_x^{(d)}\sigma_x^{(e)}\sigma_x^{(a)}\sigma_x^{(b)}\sigma_z^{(t)}$. Both branches still contain $\sigma_x^{(d)}$ and $\sigma_x^{(e)}$ (or, in the second branch, $\sigma_x^{(a)}\sigma_x^{(b)}$), and every one of those factors has expectation $0$ in the initial product state. So although the observable is no longer a single Pauli monomial, its expectation is still exactly $0$ — `c` measures independently and 50/50, the signature of $s_3 = 0$.

### Deducing $s$

Across many runs the measurements satisfy $y \cdot s = y_1 + y_2 = 0 \pmod{2}$, so `a` and `b` always agree. Possible outcomes: $(0,0,0)$, $(0,0,1)$, $(1,1,0)$, $(1,1,1)$. Seeing `a` or `b` measure 1 — always together — tells us $s_1 = s_2 = 1$. Seeing `c` measure 1 tells us $s_3 = 0$. We deduce $s = 110$.

For a full recovery of $s$, collect $n - 1 = 2$ linearly independent measurement vectors $y$ and solve the system $y \cdot s = 0$ over GF(2). For example, $(1,1,0)$ and $(0,0,1)$ give

$$s_1 + s_2 = 0, \qquad s_3 = 0$$

which together with $s \neq 000$ uniquely determines $s = 110$.